In [3]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

In [4]:
df = pd.read_csv(Path("../data/raw/chess_games.csv"))

In [6]:
rated = df[df['rated'] == True]['turns']
unrated = df[df['rated'] == False]['turns']

print(f"Rated:   n={len(rated)}, mean={rated.mean():.1f}")    # mean=62.3
print(f"Unrated: n={len(unrated)}, mean={unrated.mean():.1f}")  # mean=51.7

Rated:   n=16155, mean=62.0
Unrated: n=3903, mean=54.3


In [7]:
# Welch's t-test (no equal-variance assumption — safer default)
t_stat, p_value = stats.ttest_ind(rated, unrated, equal_var=False)
print(f"t={t_stat:.3f}, p={p_value:.6f}")   # t≈10.5, p≈0.000000

t=13.279, p=0.000000


In [ ]:
# p < 0.05 → reject H0 → rated games ARE significantly longer
# Effect size — Cohen's d
pooled_std = np.sqrt((rated.std()**2 + unrated.std()**2) / 2)
cohen_d = (rated.mean() - unrated.mean()) / pooled_std
print(f"Cohen's d: {cohen_d:.3f}")   # ≈ 0.23 — SMALL effect

Cohen's d: 0.233


In [9]:
# Real, but modest: ~10 extra turns on average
# Non-parametric alternative (our data isn't normal)
u, p_mw = stats.mannwhitneyu(rated, unrated, alternative='two-sided')

print(f"Mann-Whitney p={p_mw:.6f}")  # confirms same conclusion

Mann-Whitney p=0.000000


Null Hypothesis: White has a 50% chance of winning.

In [23]:
df['white_wins'] = (df['winner'] == 'White').astype(int)
round(df['white_wins'].mean(), 4)

np.float64(0.4986)

In [26]:
t_stat, p_value = stats.ttest_1samp(df['white_wins'], popmean=0.5)
p_value

np.float64(0.6925529790870815)

In [ ]:
WHO_URL = 'https://github.com/Priyankkoul/Life-Expectancy-WHO---Data-Analytics/blob/master/DATASET.csv?raw=true'
who = pd.read_csv(WHO_URL)
print("Shape:", who.shape)
# first_5_checks(who, "WHO")

In [ ]:
from scipy.stats import t as t_dist

def confidence_interval(data, confidence=0.95):
    n = len(data); mean = data.mean(); se = stats.sem(data)
    h = se * t_dist.ppf((1 + confidence) / 2., n - 1)
    return mean - h, mean + h

rated_turns = df[df['rated'] == True]['turns']
lo, hi = confidence_interval(rated_turns)
print(f"95% CI: ({lo:.1f}, {hi:.1f})")   # (61.8, 62.8) — narrow, n=16182

# Bootstrap CI — no normality assumption required
def bootstrap_ci(data, n_boot=1000, confidence=0.95):
    rng = np.random.default_rng(42)
    means = [rng.choice(data, len(data), replace=True).mean() for _ in range(n_boot)]
    a = (1 - confidence) / 2
    return np.percentile(means, [a*100, (1-a)*100])

lo_b, hi_b = bootstrap_ci(rated_turns.values)

print(f"Bootstrap 95% CI: ({lo_b:.1f}, {hi_b:.1f})")  # very close to parametric






95% CI: (61.4, 62.5)
Bootstrap 95% CI: (61.4, 62.5)
